In [4]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import numpy as np
import shoji
import collections
from typing import List, Optional
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import scipy as sci
import scanpy as sc

In [5]:
db = shoji.connect()

In [6]:
#Load datasat as workspace that was created for fig 1 (Figure_1A_B_Extended_Data_Figure_1C_D_E_F_G.ipynb)
ws = db.builds.jesper.h5ad.GBM_SL057

In [7]:
#Load datasat as workspace that was created in Extended fig 2 (Extended_Data_Figure_2A_C_D_E.ipynb)
ws1 = db.builds.jesper.h5ad.GBM_SL040

# WR gene module 

### Load h5ad and prepare SL057

In [8]:
adata = sc.read_h5ad('/home/jesper/data/SL057.h5ad')
adata

AnnData object with n_obs × n_vars = 123236 × 59480
    obs: 'CellCycleFraction', 'Celltype', 'Chemistry', 'Clusters', 'ClustersProbability', 'ClustersSecondary', 'ClustersSecondaryProbability', 'DoubletFlag', 'DoubletScore', 'Location', 'ManualAnnotationSL057_MAC', 'ManualAnnotationSL057_Myeloid', 'ManualAnnotationSL057_TAM', 'MitoFraction', 'NGenes', 'NeftelClass', 'Sample', 'SampleID', 'TotalUMIs', 'UnsplicedFraction', 'ValidCells', 'Zone', 'Aneuploid', 'ManualAnnotationSL057'
    var: 'Chromosome', 'End', 'Gene', 'GeneNonzeros', 'GeneTotalUMIs', 'SelectedFeatures', 'Start', 'StdevExpression', 'ValidGenes', 'Accession'
    obsm: 'Embedding', 'Factors'
    layers: 'Expression'

In [9]:
adata.var = adata.var.set_index('Gene')

In [10]:
adata.var.index = adata.var.index.astype('object')

In [11]:
adata.var_names_make_unique()

In [12]:
adata.var.index.is_unique

True

### Normalize and logaritmize

In [13]:
sc.pp.normalize_total(adata)

In [14]:
sc.pp.log1p(adata)

### Load h5ad and prepare SL040

In [15]:
adata40 = sc.read_h5ad('/home/jesper/data/SL040.h5ad')
adata40

AnnData object with n_obs × n_vars = 135482 × 59480
    obs: 'CellCycleFraction', 'Chemistry', 'Clusters', 'ClustersProbability', 'ClustersSecondary', 'ClustersSecondaryProbability', 'Direction', 'DoubletFlag', 'DoubletScore', 'Fluorescence', 'Location', 'ManualAnnotationSL040_TAM', 'ManualAnnotationSL040_central_peri', 'MitoFraction', 'NGenes', 'NeftelClass', 'Sample', 'SampleID', 'SignatureBootstrap', 'TotalUMIs', 'UnsplicedFraction', 'ValidCells', 'Zone', 'Aneuploid', 'ManualAnnotationSL040', 'ManualAnnotationSL040_Myeloid'
    var: 'Chromosome', 'End', 'Gene', 'GeneNonzeros', 'GeneTotalUMIs', 'SelectedFeatures', 'Start', 'StdevExpression', 'ValidGenes', 'Accession'
    obsm: 'Embedding', 'Factors'
    layers: 'Expression'

In [16]:
adata40.var.index
adata40.var.Gene
adata40.var = adata40.var.set_index('Gene')
adata40.var.index = adata40.var.index.astype('object')
adata40.var_names_make_unique()
adata40.var.index.is_unique


True

### Normalize and logaritmize

In [17]:
sc.pp.normalize_total(adata40)
sc.pp.log1p(adata40)

### Rank genes central/bulk vs periphery 

In [18]:
#Rank genes in oligodendrocytes for SL057 and SL040 
cells = adata[adata.obs['ManualAnnotationSL057'] == 'Oligodendrocytes',:]
sc.tl.rank_genes_groups(cells, 'Zone', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_oligoCentralPeri = sc.get.rank_genes_groups_df(cells, group='Central')[['names','logfoldchanges','pvals', 'pvals_adj']]
cells = adata40[adata40.obs['ManualAnnotationSL040'] == 'Oligodendrocytes',:]
sc.tl.rank_genes_groups(cells, 'ManualAnnotationSL040_central_peri', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_Oligo40central = sc.get.rank_genes_groups_df(cells, group=['Central'])[['names','logfoldchanges','pvals', 'pvals_adj']]

In [19]:
#Rank genes in tumor cells for SL057 and SL040
cells = adata[adata.obs['ManualAnnotationSL057'] == 'Tumor',:]
sc.tl.rank_genes_groups(cells, 'Zone', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_tumorCentralPeri = sc.get.rank_genes_groups_df(cells, group='Central')[['names','logfoldchanges','pvals', 'pvals_adj']]
cells = adata40[adata40.obs['ManualAnnotationSL040'] == 'Tumor',:]
sc.tl.rank_genes_groups(cells, 'ManualAnnotationSL040_central_peri', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_tumor40central = sc.get.rank_genes_groups_df(cells, group=['Central'])[['names','logfoldchanges','pvals', 'pvals_adj']]

In [20]:
#Rank genes in TAMs for SL057 and SL040
cells = adata[adata.obs['ManualAnnotationSL057_TAM'] == 'TAM',:]
sc.tl.rank_genes_groups(cells, 'Zone', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_TAMCentralPeri = sc.get.rank_genes_groups_df(cells, group='Central')[['names','logfoldchanges','pvals', 'pvals_adj']]
cells = adata40[adata40.obs['ManualAnnotationSL040_TAM'] == 'TAM',:]
sc.tl.rank_genes_groups(cells, 'ManualAnnotationSL040_central_peri', groups=['Central'], reference='Periphery', method="wilcoxon")
gene_rank_TAM40central = sc.get.rank_genes_groups_df(cells, group=['Central'])[['names','logfoldchanges','pvals', 'pvals_adj']]

In [21]:
%%time
#Save ranking of genes across cell types and samples
Genes = np.array(adata.var.index)
Rank = []
Rank2 = []
for x in Genes:
    A = np.where(x == np.array(gene_rank_Oligo40central['names']))[0][0]
    B = np.where(x == np.array(gene_rank_oligoCentralPeri['names']))[0][0]
    C = np.where(x == np.array(gene_rank_tumor40central['names']))[0][0]
    D = np.where(x == np.array(gene_rank_tumorCentralPeri['names']))[0][0]
    E = np.where(x == np.array(gene_rank_TAM40central['names']))[0][0]
    F = np.where(x == np.array(gene_rank_TAMCentralPeri['names']))[0][0]
    #Calculate mean rank for a final "Rank" presented as the WR gene module
    Rank.append(np.mean([A, B, C, D, E, F]))
    Rank2.append([A, B, C, D, E, F])

CPU times: user 9min 29s, sys: 224 ms, total: 9min 30s
Wall time: 9min 31s


In [32]:
#Final ranking of genes with the top ranking genes first. The value before the genes is equal to the mean ranking across 
#cell types and samples
rank = dict(zip(Rank, Genes))
sorted_rank2 = dict(sorted(rank.items()))
sorted_rank2

{13.0: 'VIM',
 15.166666666666666: 'APOE',
 15.833333333333334: 'GAPDH',
 17.166666666666668: 'MT2A',
 27.0: 'LDHA',
 31.666666666666668: 'MT-RNR1',
 34.5: 'LGALS1',
 42.666666666666664: 'TIMP1',
 44.0: 'S100A6',
 55.166666666666664: 'PKM',
 58.0: 'CHI3L1',
 63.5: 'ALDOA',
 64.66666666666667: 'SPP1',
 76.16666666666667: 'ANXA2',
 83.66666666666667: 'TPI1',
 85.16666666666667: 'CD63',
 100.33333333333333: 'APOC1',
 104.83333333333333: 'MT-RNR2',
 139.0: 'SEC61G',
 158.16666666666666: 'MYL6',
 171.16666666666666: 'PGK1',
 177.16666666666666: 'C1QB',
 179.66666666666666: 'HMOX1',
 180.0: 'CTSB',
 181.16666666666666: 'CD44',
 187.66666666666666: 'ACTB',
 196.66666666666666: 'PFN1',
 206.66666666666666: 'VEGFA',
 208.33333333333334: 'MTATP6P1',
 210.5: 'SOCS3',
 220.66666666666666: 'CLU',
 247.83333333333334: 'TAGLN2',
 252.16666666666666: 'MT1X',
 252.83333333333334: 'TMBIM6',
 256.6666666666667: 'NUPR1',
 259.5: 'GUK1',
 272.5: 'COPE',
 276.1666666666667: 'SERF2',
 289.5: 'PRDX4',
 297.66

In [29]:
#save genes only, sorted by ranking 
Genes2 = list(sorted_rank2.values())
Genes2[:50]

['VIM',
 'APOE',
 'GAPDH',
 'MT2A',
 'LDHA',
 'MT-RNR1',
 'LGALS1',
 'TIMP1',
 'S100A6',
 'PKM',
 'CHI3L1',
 'ALDOA',
 'SPP1',
 'ANXA2',
 'TPI1',
 'CD63',
 'APOC1',
 'MT-RNR2',
 'SEC61G',
 'MYL6',
 'PGK1',
 'C1QB',
 'HMOX1',
 'CTSB',
 'CD44',
 'ACTB',
 'PFN1',
 'VEGFA',
 'MTATP6P1',
 'SOCS3',
 'CLU',
 'TAGLN2',
 'MT1X',
 'TMBIM6',
 'NUPR1',
 'GUK1',
 'COPE',
 'SERF2',
 'PRDX4',
 'CIB1',
 'FTH1',
 'EIF4A1',
 'TXNDC17',
 'MYDGF',
 'GNG5',
 'IGFBP2',
 'PHPT1',
 'MIF',
 'FTL',
 'P4HB']

In [24]:
Genes2 = np.array(Genes2)
Genes2

array(['VIM', 'APOE', 'GAPDH', ..., 'DDX17', 'CLASP2', 'FCHSD2'],
      dtype='<U22')

In [25]:
#Save only the top 500 genes with the highest ranking genes first.  
Top500 = Genes2[:500]

In [26]:
Top500

array(['VIM', 'APOE', 'GAPDH', 'MT2A', 'LDHA', 'MT-RNR1', 'LGALS1',
       'TIMP1', 'S100A6', 'PKM', 'CHI3L1', 'ALDOA', 'SPP1', 'ANXA2',
       'TPI1', 'CD63', 'APOC1', 'MT-RNR2', 'SEC61G', 'MYL6', 'PGK1',
       'C1QB', 'HMOX1', 'CTSB', 'CD44', 'ACTB', 'PFN1', 'VEGFA',
       'MTATP6P1', 'SOCS3', 'CLU', 'TAGLN2', 'MT1X', 'TMBIM6', 'NUPR1',
       'GUK1', 'COPE', 'SERF2', 'PRDX4', 'CIB1', 'FTH1', 'EIF4A1',
       'TXNDC17', 'MYDGF', 'GNG5', 'IGFBP2', 'PHPT1', 'MIF', 'FTL',
       'P4HB', 'C4orf3', 'SOD2', 'DAD1', 'ENO1', 'TMED9', 'NDUFS5',
       'ATP6V0E1', 'PSMD8', 'SERPINA3', 'PPIB', 'BCAP31', 'PRELID1',
       'SPATA13', 'TNFRSF12A', 'TMEM208', 'PSME2', 'CFL1', 'B2M',
       'SLC2A3', 'SH3BGRL3', 'APOC2', 'TMSB10', 'VAMP5', 'HSPE1',
       'CHCHD2', 'LUCAT1', 'C1QC', 'MAP1B', 'SERPINE1', 'MYL12A', 'TXN',
       'UBL5', 'MT-ND4L', 'LAPTM4A', 'METRN', 'OAZ1', 'PRDX5', 'EIF5A',
       'CTSZ', 'PSMA7', 'ARPC1B', 'GBP2', 'VDAC1', 'S100A10', 'POMP',
       'C1QA', 'HSPA5', 'NUCB1', 'BSG'